# __FINAL PROJECT PHASE II__

**Project:** Do SEC insider disclosures predict abnormal stock returns?

**Group members:** [Add names]

*Raw SEC, historical membership, and index-benchmark loading are set up. Cleaning, plots, and findings remain to be completed.*

# __RESEARCH QUESTIONS__

Do publicly disclosed insider purchases and sales predict abnormal stock returns?
Over what horizon does any predictive relationship appear?

Does the relationship differ by insider role, trade size, or whether the trade was
made under a pre-scheduled plan?

We will refine the scope after checking the available sample. An abnormal return
is the difference between a stock's actual return and its expected return given
its historical relationship with the market.

### Importing

We use SciPy, NumPy, DuckDB, and Seaborn. Pandas reads and holds the data tables,
and Matplotlib supports the plots. These imports do not download any data.

In [1]:
# Imports and plot settings
import numpy as np
import pandas as pd
from scipy import stats
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### Data overview

The local inputs currently include:

- **SEC insider transaction data:** filing metadata, nonderivative transactions,
  reporting owners, and footnotes from all 24 quarters in **2020-2025**.
- **Historical S&P 500 membership:** dated index-membership snapshots, used later
  to select companies that belonged to the index at each disclosure date.
- **S&P 500 index benchmark:** the supplied CSV of daily `^SPX` index values.

**Individual-company stock prices:** still needed. The proposal plans to use
Tiingo for company prices and SPY adjusted prices; the supplied `^SPX` series is
loaded separately and is not silently substituted for those inputs.

**SEC source:** https://www.sec.gov/data-research/sec-markets-data/insider-transactions-data-sets

**Membership source:** https://github.com/fja05680/sp500

**Planned price-source documentation:** https://www.tiingo.com/documentation/end-of-day

**Final company sample and retained observation counts:** [Add after cleaning and matching]

**Shared raw-dataset link for reviewers:** [Add project data link]

### Loading the SEC data

Our local SEC data covers **2020 Q1 through 2025 Q4**, in 24 quarter folders under
`data/raw/sec/`. For each quarter, we read `SUBMISSION`, `NONDERIV_TRANS`,
`REPORTINGOWNER`, and `FOOTNOTES` as tab-separated files using `pd.read_csv`.
We combine matching tables with `pd.concat` and add `SOURCE_QUARTER` so every row
can be traced back to its original quarter.

All fields are initially read as strings to preserve identifiers and inspect the
raw values before cleaning. Optional fields present in only some quarters are
retained, with missing values elsewhere. The raw TSV files are unchanged.

Run the cell below once after opening a fresh kernel. It prints progress as each
combined table finishes. The four table types total approximately **1.31 GB** on
disk before loading into memory. Stock-price files will be added separately.

In [2]:
# Load the four SEC tables from all 24 quarters, preserving their source quarter.
from pathlib import Path
from time import perf_counter

sec_dir = Path("data/raw/sec")
if not sec_dir.is_dir():
    sec_dir = Path("insider-trading-phase2/data/raw/sec")  # If opened from the repo root.

quarter_folders = [
    sec_dir / f"{year}q{quarter}_form345"
    for year in range(2020, 2026)
    for quarter in range(1, 5)
]
table_names = ["SUBMISSION", "NONDERIV_TRANS", "REPORTINGOWNER", "FOOTNOTES"]
missing_files = [str(folder / f"{table}.tsv")
                 for folder in quarter_folders for table in table_names
                 if not (folder / f"{table}.tsv").is_file()]
if missing_files:
    raise FileNotFoundError("Check the SEC folder location or missing files:\n" + "\n".join(missing_files))

sec_tables = {}
file_counts = []
load_start = perf_counter()
for table in table_names:
    quarter_frames = []
    for folder in quarter_folders:
        frame = pd.read_csv(folder / f"{table}.tsv", sep="\t", dtype="string")
        frame["SOURCE_QUARTER"] = folder.name.split("_")[0]
        file_counts.append({"quarter": frame["SOURCE_QUARTER"].iloc[0] if len(frame) else folder.name.split("_")[0],
                            "table": table, "rows": len(frame)})
        quarter_frames.append(frame)
    sec_tables[table] = pd.concat(quarter_frames, ignore_index=True)
    del quarter_frames, frame
    print(f"{table}: {len(sec_tables[table]):,} rows loaded", flush=True)

df_submission = sec_tables["SUBMISSION"]
df_transactions = sec_tables["NONDERIV_TRANS"]
df_owners = sec_tables["REPORTINGOWNER"]
df_footnotes = sec_tables["FOOTNOTES"]
df_file_counts = pd.DataFrame(file_counts)
print(f"Loaded {len(file_counts)} files from {len(quarter_folders)} quarters in {perf_counter() - load_start:.1f} seconds.")

SUBMISSION: 1,211,888 rows loaded
NONDERIV_TRANS: 1,841,674 rows loaded
REPORTINGOWNER: 1,323,957 rows loaded
FOOTNOTES: 2,865,764 rows loaded
Loaded 96 files from 24 quarters in 12.5 seconds.


### Initial inspection

The summary below reports the dimensions of each combined table and the row counts
for every quarter. Use these checks to look for empty or incomplete inputs before
cleaning. `df_transactions.head()` displays a few raw transaction rows; these
counts include all nonderivative transaction codes and have not been filtered to P/S.

In [3]:
# Inspect combined table sizes, source-quarter coverage, and raw transaction fields.
from IPython.display import display

summary = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": frame.shape[1],
     "quarters": frame["SOURCE_QUARTER"].nunique()}
    for name, frame in sec_tables.items()
])
display(summary)
display(df_file_counts.pivot(index="quarter", columns="table", values="rows"))
print("Transaction columns:", df_transactions.columns.tolist())
display(df_transactions.head())

,table,rows,columns,quarters
0,SUBMISSION,1211888,15,24
1,NONDERIV_TRANS,1841674,29,24
2,REPORTINGOWNER,1323957,14,24
3,FOOTNOTES,2865764,4,24


table,FOOTNOTES,NONDERIV_TRANS,REPORTINGOWNER,SUBMISSION
quarter,,,,
2020q1,159716,105852,73693,69408
2020q2,113675,70429,55817,51459
2020q3,92931,60619,43766,39096
2020q4,108077,72973,49045,43607
2021q1,188895,125620,81046,74944
2021q2,140976,89108,66036,58798
2021q3,111270,74002,50774,44057
2021q4,118799,80096,52858,46675
2022q1,171741,110327,76260,71395


Transaction columns: ['ACCESSION_NUMBER', 'NONDERIV_TRANS_SK', 'SECURITY_TITLE', 'SECURITY_TITLE_FN', 'TRANS_DATE', 'TRANS_DATE_FN', 'DEEMED_EXECUTION_DATE', 'DEEMED_EXECUTION_DATE_FN', 'TRANS_FORM_TYPE', 'TRANS_CODE', 'EQUITY_SWAP_INVOLVED', 'EQUITY_SWAP_TRANS_CD_FN', 'TRANS_TIMELINESS', 'TRANS_TIMELINESS_FN', 'TRANS_SHARES', 'TRANS_SHARES_FN', 'TRANS_PRICEPERSHARE', 'TRANS_PRICEPERSHARE_FN', 'TRANS_ACQUIRED_DISP_CD', 'TRANS_ACQUIRED_DISP_CD_FN', 'SHRS_OWND_FOLWNG_TRANS', 'SHRS_OWND_FOLWNG_TRANS_FN', 'VALU_OWND_FOLWNG_TRANS', 'VALU_OWND_FOLWNG_TRANS_FN', 'DIRECT_INDIRECT_OWNERSHIP', 'DIRECT_INDIRECT_OWNERSHIP_FN', 'NATURE_OF_OWNERSHIP', 'NATURE_OF_OWNERSHIP_FN', 'SOURCE_QUARTER']


,ACCESSION_NUMBER,NONDERIV_TRANS_SK,SECURITY_TITLE,SECURITY_TITLE_FN,TRANS_DATE,TRANS_DATE_FN,DEEMED_EXECUTION_DATE,DEEMED_EXECUTION_DATE_FN,TRANS_FORM_TYPE,TRANS_CODE,...,TRANS_ACQUIRED_DISP_CD_FN,SHRS_OWND_FOLWNG_TRANS,SHRS_OWND_FOLWNG_TRANS_FN,VALU_OWND_FOLWNG_TRANS,VALU_OWND_FOLWNG_TRANS_FN,DIRECT_INDIRECT_OWNERSHIP,DIRECT_INDIRECT_OWNERSHIP_FN,NATURE_OF_OWNERSHIP,NATURE_OF_OWNERSHIP_FN,SOURCE_QUARTER
0,0001446472-20-000002,281943,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,P,...,<NA>,1537800.0,F2,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
1,0001084991-20-000014,379212,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,M,...,<NA>,8560.0,<NA>,<NA>,<NA>,I,<NA>,By Rabbi Trust,<NA>,2020q1
2,0001104659-20-041411,245379,Common Stock,<NA>,28-MAR-2020,<NA>,<NA>,<NA>,4,F,...,<NA>,54890.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
3,0001104659-20-041411,245380,Common Stock,<NA>,31-MAR-2020,<NA>,<NA>,<NA>,4,F,...,<NA>,54338.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1
4,0001104659-20-041409,272457,Common Stock,<NA>,28-MAR-2020,<NA>,<NA>,<NA>,4,F,...,<NA>,24497.0,<NA>,<NA>,<NA>,D,<NA>,<NA>,<NA>,2020q1


### Loading historical S&P 500 membership

The CSV in `data/raw/historical/` contains a `date` and a comma-separated list of
`tickers` for each membership snapshot. Read it with `pd.read_csv`; Pandas preserves
that quoted ticker list as one column.

These are dated snapshots, not a row for every trading day. When matching filings
later, use the most recent snapshot on or before the disclosure date, and handle
historical ticker changes explicitly. We are only loading and inspecting the file here.

Source: [fja05680/sp500 historical membership](https://github.com/fja05680/sp500).

In [4]:
# Load and inspect the supplied historical membership CSV.
from pathlib import Path

raw_dir = Path("data/raw")
if not raw_dir.is_dir():
    raw_dir = Path("insider-trading-phase2/data/raw")

historical_file = raw_dir / "historical" / "S&P 500 Historical Components & Changes (Updated).csv"
df_historical = pd.read_csv(historical_file)
historical_dates = pd.to_datetime(df_historical["date"], format="%Y-%m-%d")
print(f"Historical membership: {len(df_historical):,} rows, {df_historical.shape[1]} columns")
print(f"Snapshot dates: {historical_dates.min().date()} to {historical_dates.max().date()}")
print("Missing values:", df_historical.isna().sum().to_dict())
display(df_historical.head())

Historical membership: 2,720 rows, 2 columns
Snapshot dates: 1996-01-02 to 2026-08-18
Missing values: {'date': 0, 'tickers': 0}


,date,tickers
0,1996-01-02,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
1,1996-01-03,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
2,1996-01-04,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
3,1996-01-10,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
4,1996-01-11,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."


### Loading the S&P 500 index benchmark

The supplied CSV in `data/raw/prices/` contains `Dates` and
`S&P 500 (^SPX) - Index Value`. We load it as **`df_benchmark`** to distinguish this
market-index history from the individual-company prices still required.

This file is the S&P 500 price index, not a SPY adjusted-price series or a set of
company stock prices. We have not changed the planned return model or substituted
it for dividend-adjusted SPY returns. Decide and document the benchmark treatment
before calculating abnormal returns.

The code below reads the raw data and checks the date range. Date parsing is used
for inspection; the original column names and values are preserved.

**Source details:** provided CSV labelled CapIQ. Add the export source, retrieval
 date, and the team's collection method to the data description.

In [5]:
# Load and inspect the supplied index-price CSV.
benchmark_file = raw_dir / "prices" / "S&P 500_Historical Data_CapIQ - Sheet1.csv"
df_benchmark = pd.read_csv(benchmark_file)
benchmark_dates = pd.to_datetime(df_benchmark["Dates"], format="%b-%d-%Y")
print(f"S&P 500 index benchmark: {len(df_benchmark):,} rows, {df_benchmark.shape[1]} columns")
print(f"Price dates: {benchmark_dates.min().date()} to {benchmark_dates.max().date()}")
print("Missing values:", df_benchmark.isna().sum().to_dict())
display(df_benchmark.head())

S&P 500 index benchmark: 25,598 rows, 2 columns
Price dates: 1928-01-03 to 2026-10-15
Missing values: {'Dates': 0, 'S&P 500 (^SPX) - Index Value': 0}


,Dates,S&P 500 (^SPX) - Index Value
0,Jan-03-1928,17.76
1,Jan-04-1928,17.72
2,Jan-05-1928,17.55
3,Jan-06-1928,17.66
4,Jan-07-1928,17.68


# __DATA COLLECTION AND CLEANING APPENDIX__

### Collection process

[Describe which SEC quarters and stock-price histories were obtained, how they
were downloaded, and when. Document the company-selection rule and the raw filenames.
Describe collection in text; include the cleaning code below.]

### Cleaning historical S&P 500 membership

We clean the membership data separately from the SEC transactions. The original
`df_historical` and source CSV are preserved.

1. Parse the snapshot dates and normalize ticker whitespace/capitalization.
2. Remove repeated tickers within a snapshot and exact duplicate snapshots.
   Stop for missing values, empty membership lists, or conflicting snapshots on
   the same date, because silently dropping them could create incorrect coverage.
3. Derive the study boundaries from `FILING_DATE` in the loaded SEC batches
   (`2020q1` through `2025q4`), rather than treating a trading calendar as the
   source of the quarter definitions.
4. Keep the latest snapshot on or before the first filing date, plus all subsequent
   snapshots through the last filing date. The starting snapshot can be dated
   before 2020 because its membership remains in effect until the next change.

We initially treat the community dataset's snapshot date as the date on which its
membership takes effect. Confirm that convention against a sample of original
index announcements before using it for inference. Ticker punctuation is preserved;
this step does not resolve ticker renames or map symbols to a price provider.

In [6]:
# Clean historical membership snapshots and retain coverage for the actual SEC filing window.
filing_dates = pd.to_datetime(df_submission["FILING_DATE"], format="mixed", errors="raise")
if filing_dates.isna().any() or filing_dates.empty:
    raise ValueError("Review missing SEC filing dates before defining the study window.")
study_start = filing_dates.min().normalize()
study_end = filing_dates.max().normalize()
study_end_exclusive = study_end + pd.Timedelta(days=1)

historical = df_historical[["date", "tickers"]].copy()
if historical.isna().any().any():
    raise ValueError("Review missing snapshot dates or ticker lists before cleaning.")
historical["date"] = pd.to_datetime(historical["date"], format="%Y-%m-%d", errors="raise")
ticker_lists = historical["tickers"].astype("string").str.split(",")
ticker_lists = ticker_lists.map(lambda values: [ticker.strip().upper() for ticker in values if ticker.strip()])
repeated_tickers = int(ticker_lists.map(lambda values: len(values) - len(set(values))).sum())
if ticker_lists.map(len).eq(0).any():
    raise ValueError("An empty membership list requires review.")
historical["tickers"] = ticker_lists.map(lambda values: ",".join(sorted(set(values))))
exact_duplicates = int(historical.duplicated(["date", "tickers"]).sum())
historical = historical.drop_duplicates(["date", "tickers"]).sort_values("date")
if historical["date"].duplicated().any():
    raise ValueError("Conflicting membership snapshots share the same date; review them first.")

baseline = historical.loc[historical["date"].le(study_start)].tail(1)
if baseline.empty:
    raise ValueError("A membership snapshot on or before the first filing is required.")
changes = historical.loc[historical["date"].gt(study_start) & historical["date"].le(study_end)]
historical_snapshots = pd.concat([baseline, changes], ignore_index=True)
historical_snapshots = historical_snapshots.rename(columns={"date": "snapshot_date"})
historical_snapshots["valid_from"] = historical_snapshots["snapshot_date"].clip(lower=study_start)
historical_snapshots["valid_to"] = historical_snapshots["snapshot_date"].shift(-1).fillna(study_end_exclusive)

print(f"SEC filing window: {study_start.date()} through {study_end.date()}")
print(f"Starting membership snapshot: {baseline['date'].iloc[0].date()}")
print(f"Original snapshots: {len(df_historical):,}; retained snapshots: {len(historical_snapshots):,}")
print(f"Exact duplicate snapshots removed: {exact_duplicates}; repeated ticker entries removed: {repeated_tickers}")
display(historical_snapshots[["snapshot_date", "valid_from", "valid_to"]].head())

SEC filing window: 2020-01-02 through 2025-12-31
Starting membership snapshot: 2019-12-23
Original snapshots: 2,720; retained snapshots: 91
Exact duplicate snapshots removed: 0; repeated ticker entries removed: 0


,snapshot_date,valid_from,valid_to
0,2019-12-23,2020-01-02,2020-01-28
1,2020-01-28,2020-01-28,2020-03-03
2,2020-03-03,2020-03-03,2020-04-03
3,2020-04-03,2020-04-03,2020-04-06
4,2020-04-06,2020-04-06,2020-05-12


### One ticker per membership interval

We split each retained snapshot into one row per ticker. The cleaned table
`df_historical_clean` contains:

| Column | Meaning |
|---|---|
| `ticker` | Symbol recorded in the historical membership source |
| `snapshot_date` | Date of the source snapshot |
| `valid_from` | First included date in this study window |
| `valid_to` | First date no longer covered by this snapshot; **exclusive** |

A disclosure matches a row when `valid_from <= FILING_DATE < valid_to` and the
company symbol has been mapped correctly. For example, `valid_to = 2026-01-01`
includes December 31, 2025. These are **snapshot coverage intervals**, not claimed
original company admission or exit dates. A company remaining in the index across
several snapshots will correctly have several adjacent rows.

In [7]:
# Expand the retained snapshots into a tidy ticker-by-interval table.
df_historical_clean = historical_snapshots.assign(
    ticker=historical_snapshots["tickers"].str.split(",")
).explode("ticker", ignore_index=True)
df_historical_clean = df_historical_clean[
    ["ticker", "snapshot_date", "valid_from", "valid_to"]
].sort_values(["ticker", "valid_from"]).reset_index(drop=True)

# Check interval integrity before this table is used for any joins.
assert not df_historical_clean.isna().any().any()
assert not df_historical_clean.duplicated(["ticker", "valid_from"]).any()
assert df_historical_clean["valid_from"].lt(df_historical_clean["valid_to"]).all()
assert df_historical_clean["valid_from"].ge(study_start).all()
assert df_historical_clean["valid_to"].le(study_end_exclusive).all()
next_start = df_historical_clean.groupby("ticker")["valid_from"].shift(-1)
assert (next_start.isna() | df_historical_clean["valid_to"].le(next_start)).all()

print(f"Cleaned membership rows: {len(df_historical_clean):,}")
print(f"Distinct ticker symbols during the study window: {df_historical_clean['ticker'].nunique():,}")
print("Ticker counts are not necessarily unique company counts: symbols and share classes can change.")
display(df_historical_clean.head(10))

Cleaned membership rows: 45,850
Distinct ticker symbols during the study window: 627
Ticker counts are not necessarily unique company counts: symbols and share classes can change.


,ticker,snapshot_date,valid_from,valid_to
0,A,2019-12-23,2020-01-02,2020-01-28
1,A,2020-01-28,2020-01-28,2020-03-03
2,A,2020-03-03,2020-03-03,2020-04-03
3,A,2020-04-03,2020-04-03,2020-04-06
4,A,2020-04-06,2020-04-06,2020-05-12
5,A,2020-05-12,2020-05-12,2020-05-22
6,A,2020-05-22,2020-05-22,2020-06-22
7,A,2020-06-22,2020-06-22,2020-09-18
8,A,2020-09-18,2020-09-18,2020-09-21
9,A,2020-09-21,2020-09-21,2020-10-07


### Cleaning the SEC transactions


[Explain each cleaning step immediately before its code cell. Likely steps include
selecting Form 4 transactions, filtering purchase/sale codes, parsing dates and
numeric values, reviewing missing values, and resolving amendments.]

P and S codes can include open-market or private transactions. Confirm the final
security and transaction eligibility rules with the actual records.

In [8]:
# Add the SEC cleaning steps after inspecting the supplied files.

### Joining the SEC tables

[Describe which columns connect the tables and why the join preserves the intended
unit of observation. Use DuckDB for the joins if appropriate.]

One filing can contain multiple transactions and reporting owners. Check for
duplicated transactions after joining; a direct owner-table join can multiply rows.

In [9]:
# Add the DuckDB joins after checking the actual keys and table structure.

### Preparing prices and matching companies

[Describe the ticker/date matching, date conversion, missing-price checks, and
split/dividend adjustments. Document symbol changes and unmatched companies.]

Use the filed shares and transaction price to calculate reported trade value.
Use adjusted stock prices for returns. Verify the disclosure timing and trading
calendar before assigning event windows.

In [10]:
# Add the price-cleaning and matching steps after the price data is supplied.

### Saving intermediate and cleaned data

Save the cleaned historical membership table to
`data/processed/historical_membership_clean.csv`. This contains the ticker/date
intervals needed for later matching; the original membership and SEC files remain
separate. CSV does not retain date types, so parse `snapshot_date`, `valid_from`,
and `valid_to` as dates if this saved file is loaded in a later session.

[Document additional intermediate files here as the SEC and price cleaning proceeds.]

In [11]:
# Save the cleaned historical membership intervals for reuse.
processed_dir = raw_dir.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
df_historical_clean.to_csv(processed_dir / "historical_membership_clean.csv", index=False)
print("Saved data/processed/historical_membership_clean.csv")

Saved data/processed/historical_membership_clean.csv


# __EXPLORATORY DATA ANALYSIS__

### Summary statistics

[Report sample size, company counts, date coverage, missingness, purchase/sale
counts, and typical trade sizes. Explain what each summary tells us.]

In [12]:
# Add summary statistics using the real cleaned dataset.

### Plots

Possible initial plots include purchases versus sales, trade-size distributions,
filing counts over time, and the gap between trade dates and filing dates.

[Choose plots based on the actual data. Put a brief explanation before each plot
and an interpretation after it. Use Seaborn and NumPy/SciPy where useful.]

Do not run the preregistered hypothesis tests before the preregistrations are finalized.

In [13]:
# Add focused Seaborn plots after the data has been cleaned.

### What we learned from the EDA

[Describe findings using actual numbers and plots. Explain whether the research
questions or dataset description should change based on what the EDA reveals.]

# __DATA DESCRIPTION__

### Motivation

**Why was this dataset created?**

The combined dataset will let us examine whether public insider disclosures
predict stock returns beyond what would normally be expected from market movements.
The SEC source records regulatory disclosures; the price source records market outcomes.

### Composition

**What do the observations and attributes represent?**

[Describe the actual analysis-ready rows and columns. Distinguish transaction rows
from any company/disclosure-day events created for analysis. Add a data dictionary.]

**How many observations are included, and what period do they cover?**

[Complete after cleaning.]

### Collection and preprocessing

**How was the combined dataset created?**

[Summarize the sources, tools, selection criteria, cleaning, joins, and transformations
actually used. Refer to the cleaning appendix.]

**What influenced which records were observed?**

[Explain reporting requirements, company selection, and price availability.]

**If people are represented, what was the purpose of the original collection?**

The SEC records concern corporate insiders whose transactions and holdings are
subject to regulatory disclosure. Filing these records does not establish consent
to every downstream interpretation. Avoid inferring misconduct from an association.

**Where can the original data be found?**

[Add source links and the course-approved shared raw-data location.]

# __DATA LIMITATIONS__

Potential limitations to evaluate with the actual data:

1. **Membership coverage and sample selection:** historical membership can reduce
   selection based on current constituents, but the community list may contain errors
   and still requires date/symbol validation. An S&P 500 sample mainly represents large firms.
2. **Missing and changing symbols:** a historical ticker may not match the vendor's
   symbol, and delisted or acquired companies may lack complete histories.
3. **Disclosure timing:** transaction dates differ from public disclosure dates;
   event windows must use information available at the assumed trading time.
4. **Amendments and joint filings:** corrections and multiple reporting owners can
   create duplicates or incorrect attribution if handled carelessly.
5. **Interpretation:** sales may reflect personal liquidity or diversification.
   Earnings and other news can explain returns. Associations do not establish causation.
6. **People and impact:** results should not label individuals as engaging in illegal
   activity or be presented as guaranteed trading profits.

[Add measured data gaps and explain how each relevant limitation changes what
the project can conclude.]

# __PREREGISTRATION STATEMENT 1__

**Draft hypothesis:** Public insider purchase disclosures are followed by positive
average short-horizon cumulative abnormal returns.

**Proposed analysis:** [Before running this test, specify the event unit, sample,
disclosure/entry rule, return horizon, market-model estimation window, exclusions,
statistical test/model, uncertainty calculation, and significance threshold.]

**Status:** Draft, not tested. Finalize and record this statement before examining
the corresponding outcome relationship. Report the final result regardless of significance.

# __PREREGISTRATION STATEMENT 2__

**Draft hypothesis:** C-suite purchase disclosures are followed by larger abnormal
returns than purchase disclosures by other insiders.

**Proposed analysis:** [Before running this test, define role categories and the
comparison sample, specify a feasible regression/model and controls, and address
joint filings, repeated companies, and uncertainty.]

**Status:** Draft, not tested. Confirm that the actual data supports this comparison,
then finalize the analysis before examining the corresponding outcome relationship.
At least one final-project model must address a research question.

# __QUESTIONS FOR REVIEWERS__

1. Does our actual sample support the proposed comparison between C-suite insiders
   and other insiders?
2. Is our event-date and trading-entry rule appropriate for the disclosure and price
   data available?
3. Are our amendment handling, missing-price rules, and company-selection decisions reasonable?
4. What changes would make the preregistered models feasible and informative using
   course methods?

[Update these questions after the group has discussed the real EDA.]